# How much does the choice of particle size definition matter?

An imaged particle has no single diameter: the package offers six definitions (length along the
flight direction, width across the array, their mean, the diagonal, the larger of the two, and the
diameter of the minimum enclosing circle). This notebook uses
`arrayprobe_enhancements.dmax_sensitivity.compute_sensitivity` to put all six side by side for every
particle and to report how far they spread. Because radar reflectivity goes as the sixth power of
diameter, the spread is also quoted as the reflectivity shift in dB it would cause. It runs in
seconds on seeded synthetic particles and needs no data files.

**Before this:** [06_read_products](../1_getting_started/06_read_products.ipynb) (Reading the output files). **Related:** [01_enclosing_circle_conventions](../4_methods_and_literature/01_enclosing_circle_conventions.ipynb). **Needs:** the enhancements package.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/6_science/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## 1. Rectangles of known shape

Each synthetic particle is a filled rectangle `L_x` by `L_y` diodes, so every definition has an
exact answer: for aspect ratio `a = L_x / L_y >= 1` the smallest diameter is `L_y` and the largest
is the diagonal, `L_y * sqrt(a^2 + 1)`, which is also the diameter of the rectangle's minimum
enclosing circle. The cell builds three aspect ratios and checks the six diameters against these
values.

In [2]:
import numpy as np
from arrayprobe_enhancements.dmax_sensitivity import compute_sensitivity

diodesize = 0.01                                     # mm per diode
L_y = np.array([10, 10, 10], dtype=np.int16)
L_x = np.array([10, 15, 20], dtype=np.int16)         # aspect ratios 1.0, 1.5, 2.0
minR = np.hypot(L_x, L_y) * diodesize                # enclosing circle of a rectangle
sens = compute_sensitivity(L_x, L_y, diodesize, minR, engine="3.5")
print(sens.d_choice_labels)
print(np.round(sens.diameters, 4))

a = L_x / L_y
expected = diodesize * L_y[:, None] * np.column_stack(
    [a, np.ones_like(a), (a + 1) / 2, np.hypot(a, 1), np.maximum(a, 1), np.hypot(a, 1)])
assert np.allclose(sens.diameters, expected, rtol=0, atol=1e-7)   # enclosing circle held at single precision
spread_db = 60 * np.log10(sens.diameters.max(axis=1) / sens.diameters.min(axis=1))
for ai, rel, zdb in zip(a, sens.spread_relative, spread_db):
    print(f"aspect {ai:.1f}: spread {100 * rel:.2f} % of the mean diameter, "
          f"{zdb:.2f} dB in reflectivity between the largest and smallest definition")
assert abs(spread_db[2] - 60 * np.log10(np.sqrt(5))) < 1e-5

('L_x', 'L_y', 'mean(L_x,L_y)', 'hypotenuse(L_x,L_y)', 'max(L_x,L_y)', 'D of minimum enclosing circle')
[[0.1    0.1    0.1    0.1414 0.1    0.1414]
 [0.15   0.1    0.125  0.1803 0.15   0.1803]
 [0.2    0.1    0.15   0.2236 0.2    0.2236]]
aspect 1.0: spread 36.40 % of the mean diameter, 9.03 dB in reflectivity between the largest and smallest definition
aspect 1.5: spread 54.39 % of the mean diameter, 15.36 dB in reflectivity between the largest and smallest definition
aspect 2.0: spread 67.59 % of the mean diameter, 20.97 dB in reflectivity between the largest and smallest definition


Even a square particle spreads: its diagonal is `sqrt(2)` times its side, so choosing the diagonal
over the side shifts reflectivity by `60 log10(sqrt(2))`, about 9 dB, for the same particle.

## 2. A population with measurement noise

A real minimum enclosing circle comes from the image outline, not from an exact rectangle. Here
400 particles take seeded random sizes and aspect ratios, and the enclosing-circle diameter carries
up to half a diode of noise. The pooled `summary()` is compared with the same statistic computed
from the noiseless diagonal; the noise touches one of six columns and averages out, so the two
agree within one percentage point rather than exactly.

In [3]:
rng = np.random.default_rng(47)
n = 400
Ly = rng.integers(5, 40, n).astype(np.int16)
Lx = np.round(Ly * rng.uniform(1.0, 2.5, n)).astype(np.int16)
diag = np.hypot(Lx, Ly) * diodesize
noisy_minR = diag + rng.uniform(-0.5, 0.5, n) * diodesize
pop = compute_sensitivity(Lx, Ly, diodesize, noisy_minR, engine="3.5")
clean = compute_sensitivity(Lx, Ly, diodesize, diag, engine="3.5")
s_noisy, s_clean = pop.summary(), clean.summary()
print({k: round(v, 3) for k, v in s_noisy.items()})
print(f"mean relative spread: {s_noisy['mean_relative_spread_pct']:.2f} % with noise, "
      f"{s_clean['mean_relative_spread_pct']:.2f} % noiseless")
assert s_noisy['n_excluded_non_finite'] == 0
assert abs(s_noisy['mean_relative_spread_pct'] - s_clean['mean_relative_spread_pct']) < 1.0

{'n_particles': 400, 'n_excluded_non_finite': 0, 'mean_relative_spread_pct': 60.709, 'max_relative_spread_pct': 80.242, 'mean_absolute_spread_mm': 0.228}
mean relative spread: 60.71 % with noise, 60.16 % noiseless


## Summary

- `compute_sensitivity` returns the six diameters for every particle, side by side, using the
  engine's own sizing code (`engine="3.5"` here), so nothing is re-derived.
- `spread_absolute`, `spread_relative` and `summary()` say how far the definitions disagree;
  particles with a non-finite diameter are counted separately rather than dropped silently.
- For the same particle the definitions differ by tens of percent, which is several dB in
  reflectivity: state the size definition with every size distribution or moment you report.